# Source Extraction

<img src="https://live.staticflickr.com/65535/54443002259_4a8e1249dd_b.jpg" alt="Embedded Photo" width="500">

*Image generated using ChatGPT.*

## Introduction

Language models are sometimes prone to saying falsehoods or half-truths, and to inventing facts without providing sources. Today, systems are increasingly used that do not answer questions directly at first, but instead search a database, such as a document collection, and only then generate an answer from the best-matching documents. Such an answer is more likely to be grounded in reality and can be verified by a human, provided that the correct sources were found.

Of course, there may be many sources, so search methods must be efficient. Processing everything "at once" directly with a language model is not feasible. In this task, you will focus on finding the best sources for a given sentence using **embeddings**.

Imagine that you are an AI engineer at a company developing a tool for scientific fact-checking. Your task is to create a module that can quickly and effectively find reliable scientific publications that support or refute specific claims. With your solution, scientists, journalists, and decision-makers will be able to verify information using solid scientific evidence, which is especially important in an age of disinformation.


## Task

Your task is to develop a system that generates high-quality vector representations, or embeddings, for both queries and source documents, enabling accurate matching of the correct sources to the queries.

You are given **queries** and a **corpus** of documents/sources. You must implement functions that assign real-valued vectors of dimension $768$ to both queries and sources. These vectors will be used to find sources for each query by the provided evaluation function, which selects the $k=10$ nearest neighbours from the document collection for a given query.

In your solution, you may use the provided model based on the GPT-2 architecture, which has been specially fine-tuned to help obtain good-quality embeddings.

While working on the solution, you can test its effectiveness on a validation set, which allows you to evaluate the quality of the generated embeddings in the context of retrieving the correct source documents.

### Data

The data available to you in this task is:

- A set of queries for which the appropriate sources must be found.
- A document corpus containing scientific publications that may be sources for the queries.
- Information about query-to-document matches in the validation set.

Your solution will be evaluated on the *SciFact* benchmark. It is used to evaluate search and fact-verification systems in a scientific context. It consists of a set of claims, called queries, based on real scientific publications, while the document database, called the corpus, contains publications from life sciences and medicine. For each claim, there is at least one publication that supports or refutes it. We provide code for loading the data, so the data is described here only for context.


**The file `corpus.jsonl`** contains unique identifiers, titles, and abstracts of scientific papers.

Example of a single document:
```
{
    "text_id": 13734012,
    "title": "Prevalent abnormal prion protein in human appendixes after bovine spongiform encephalopathy epizootic: large scale survey",
    "text": "OBJECTIVES To carry out a further survey (...) CONCLUSIONS This study corroborates previous studies and suggests a high prevalence of infection with abnormal PrP, indicating vCJD carrier status in the population compared with the 177 vCJD cases to date. These findings have important implications for the management of blood and blood products and for the handling of surgical instruments."
}
```

**The file `queries_val.jsonl`** contains claim texts and the identifier of the matching source text. The test set on which your final solution will be evaluated **will not contain** the identifiers of the matching source texts.

Example of a single query:
```
{
    "query": "1 in 5 million in UK have abnormal PrP positivity.",
    "matching_text_id": 13734012
}
```

### Evaluation Criterion

Your implemented methods `Embedder.encode_queries` and `Embedder.encode_corpus` will be used to process queries $q \in Q$ and documents $d \in C$ into vectors. Below, $q$ and $d$ may refer either to the texts or to their embeddings, depending on context.

Assume that query $q \in Q$ has a golden document $d \in C$.
The evaluation code sorts all documents by distance from $q$, obtaining documents $K_1, K_2, ..., K_n$, where $K_1$ is closest. Then $I$ denotes the index of the golden document $d$ in this sequence. That is, $I - 1$ is the number of documents whose distance from $q$ is smaller than the distance from $q$ to $d$.

Distance between vectors is computed using cosine similarity, which for vectors $v, w \in \mathbb{R}^n$ is defined as $\frac{v^Tw}{||v|| \cdot ||w||}$, where $||v||$ is the length of vector $v$.

The score for query $q$ is defined as

$$\text{nDCG@10}(q) = \begin{cases}
\frac{1}{\log_2(I + 1)} & \text{if $I \leq 10$} \\
0 & \text{otherwise.}
\end{cases}$$

So the closer the golden document is placed to the query relative to other documents, the higher the score. If 10 wrong documents are closer to the query, the score for that example is 0.

The final evaluation of your solution is based on **nDCG@10**, computed as the average value of this metric over all queries $(q \in Q)$.

- If **nDCG@10** is **lower than 0.2**, you receive **0 points**.
- If it **exceeds 0.5**, you receive the **maximum score**, **100 points**.

Scores between these thresholds are assigned proportionally.


## Constraints

- Your solution will be tested on the Competition Platform without internet access and in an environment with a GPU.
- Evaluation of your final solution on the Competition Platform may not take longer than 10 minutes with GPU.
- The embedding of each query and each text should have dimension 768.
- Allowed libraries: `torch`, `pandas`, `numpy`, `nltk`, `transformers`.

## Submission Files

Submit only this notebook completed with your solution; see the `Embedder` class.

## Hints

- GPT-2 is a decoder-type language model. Decoder models work as follows: for a given sequence of tokens, such as a prefix of the processed sentence, $t_1, t_2, \dots, t_n$, they compute a hidden vector $h_{n+1} \in \mathbb{R}^d$, and then transform it using one of their weight matrices into $p_{n+1} \in \mathbb{R}^m$, a probability distribution over vocabulary tokens.
- There are many documents compared with the available execution time.

## Evaluation

During grading, the `FINAL_EVALUATION_MODE` flag will be set to `True`.

You can earn between 0 and 100 points for this task. The number of points you receive will be calculated on a secret test set on the Competition Platform using the formula above, rounded to an integer. If your solution does not meet the criteria above or does not run correctly, you will receive 0 points for the task.


# Starter Code

In this section, we initialize the environment by importing the required libraries and functions. The prepared tokenizer, data-loading, and evaluation code will help you work with the data and solve the task.


In [1]:
######################### DO NOT CHANGE THIS CELL ##########################

FINAL_EVALUATION_MODE = False  # During grading, this value will be changed to True


In [2]:
######################### DO NOT CHANGE THIS CELL ##########################

import json
import os
from math import log2

import torch
from tqdm import tqdm
from transformers import AutoModel, AutoTokenizer


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class Tokenizer:
    def __init__(self, tokenizer_path, length=150):
        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)
        self.tokenizer.pad_token = self.tokenizer.eos_token
        self.tokenizer.padding_side = "right"
        self.length = length

    def __call__(self, batch_text):
        batch_tensor = self.tokenizer(
            batch_text,
            max_length=self.length,
            truncation=True,
            padding=True,
            return_tensors="pt"
        )
        return batch_tensor.to(device)

## Loading Data
In this part of the task, we load the training data.


In [ ]:
######################### DO NOT CHANGE THIS CELL ##########################

def load_corpus(file):
    corpus = {}
    with open(file, encoding="utf8") as f_in:
        for line in f_in:
            line = json.loads(line)
            corpus[line.get("text_id")] = {
                "text": line.get("text"),
                "title": line.get("title"),
            }
    return corpus

def load_queries(file):
    queries = {}
    matching_texts = {}
    with open(file, encoding="utf8") as f_in:
        for query_num, line in enumerate(f_in):
            line = json.loads(line)

            queries[query_num] = line.get("query")
            matching_texts[query_num] = line.get("matching_text_id")
    return queries, matching_texts

corpus = load_corpus("corpus.jsonl")
queries, matching_texts = load_queries("queries_val.jsonl")

print(f"Loaded {len(corpus)} texts and {len(queries)} queries.")

## Evaluation Criterion Code

Code similar to the following will be used to evaluate the solution on the test set.


In [4]:
######################### DO NOT CHANGE THIS CELL ##########################

def evaluate_retrieval_ndcg(
    golden_matches: dict[int, int],
    results: dict[int, dict[int, float]],
) -> float:
    """
    Compute the nDCG metric value for the given search results.

    The function computes your solution's score based on the top-k best documents according to your embedder.

    :param golden_matches: Dictionary of gold assignments, where the key is the query id and the value is the correct document id.
    :param results: Dictionary of search results, where the key is the query id and the value is a dictionary of document ids and their similarities to the query.
    :return: nDCG metric value.
    """

    for query_id, v in results.items():
        results[query_id] = {k: v for k, v in sorted(v.items(), key=lambda item: -item[1])}

    ndcg_sum = 0
    for query_id, v in results.items():
        golden_document = golden_matches[query_id]
        for i, document_id in enumerate(v.keys()):
            if golden_document == document_id:
                ndcg_sum += 1 / log2(i + 2)

    ndcg = round(ndcg_sum / len(results), 5)
    return ndcg


def compute_score(ndcg: float) -> float:
    """
    Compute the point score from the nDCG metric value.
    """
    lower_bound = 0.2
    upper_bound = 0.5

    if ndcg <= lower_bound:
        return 0
    elif lower_bound < ndcg < upper_bound:
        return int(round(100 * (ndcg - lower_bound) / (upper_bound - lower_bound)))
    else:
        return 100


### Search
Below is code used to select the top-$k$ best documents from the corpus for a given query.


In [5]:
######################### DO NOT CHANGE THIS CELL ##########################

def cos_sim(a: torch.Tensor, b: torch.Tensor):
    """
    Computes the cosine similarity cos_sim(a[i], b[j]) for all i and j.
    :return: Matrix with res[i][j]  = cos_sim(a[i], b[j])
    """
    a_norm = torch.nn.functional.normalize(a, p=2, dim=1)
    b_norm = torch.nn.functional.normalize(b, p=2, dim=1)
    return torch.mm(a_norm, b_norm.transpose(0, 1))

def search_topk_texts(
    embedder,
    corpus: dict[str, dict[str, str]],
    queries: dict[str, str],
    top_k: int = 10,
) -> dict[str, dict[str, float]]:
    results = {}

    # Create embeddings for all queries using model.encode_queries()
    # Runs semantic search against the corpus embeddings
    # Returns a ranked list with the corpus ids
    query_ids = list(queries.keys())
    results = {qid: {} for qid in query_ids}
    queries = [queries[qid] for qid in queries]
    query_embeddings = embedder.encode_queries(queries)

    corpus_ids = sorted(
        corpus,
        key=lambda k: len(corpus[k].get("title", "") + corpus[k].get("text", "")),
        reverse=True,
    )
    corpus = [corpus[cid] for cid in corpus_ids]

    # Encode chunk of corpus
    corpus_embeddings = embedder.encode_corpus(corpus)

    # Compute similarites using cosine-similarity
    cos_scores = cos_sim(query_embeddings, corpus_embeddings)
    cos_scores[torch.isnan(cos_scores)] = -1

    # Get top-k values
    cos_scores_top_k_values, cos_scores_top_k_idx = torch.topk(
        cos_scores,
        min(top_k + 1, len(cos_scores[1])),
        dim=1,
        largest=True,
        sorted=False,
    )
    cos_scores_top_k_values = cos_scores_top_k_values.cpu().tolist()
    cos_scores_top_k_idx = cos_scores_top_k_idx.cpu().tolist()

    for query_itr in range(len(query_embeddings)):
        query_id = query_ids[query_itr]
        for score, corpus_id in zip(cos_scores_top_k_values[query_itr], cos_scores_top_k_idx[query_itr]):
            results[query_id][corpus_ids[corpus_id]] = score

    return results

# Your Solution
Put your solution in this section. Make changes only here!


In [6]:
class Embedder:
    # Do not change the constructor signature.
    def __init__(self):
        # TODO: You may change this method,
        # but do not change its signature! That means do not change the arguments.
        self.model = AutoModel.from_pretrained("Muennighoff/SGPT-125M-weightedmean-msmarco-specb-bitfit")
        self.tokenizer = Tokenizer("Muennighoff/SGPT-125M-weightedmean-msmarco-specb-bitfit")

    def encode_queries(self, queries: list[str]):
        """
        Function for encoding queries.
        :param queries: List of queries to encode.
        :return: Query embeddings, a tensor with shape (n, 768), where n = len(queries) is the number of queries.
        """

        # TODO: Implement this method: encode the queries.
        # Do not change this method's signature! That means do not change the arguments.
        # Remember that you may use the HuggingFace GPT-2 model.
        # You may use the Tokenizer implemented near the top of the notebook.
        # Hint: Evaluation will be faster if the returned tensor is located on the GPU.
        ...
        return torch.ones(len(queries), 768).to(device)

    def encode_corpus(self, texts: list[dict]):
        """
        Function for encoding source texts.
        :param texts: List of texts to encode. Each text is represented as a dictionary:
            {
                "title": "...",
                "text": "...",
            }
        :return: Text embeddings, a tensor with shape (m, 768), where m = len(texts) is the number of texts.
        """

        # TODO: Implement this method: encode the source texts.
        # Do not change this method's signature! That means do not change the arguments.
        ...
        return torch.ones(len(texts), 768).to(device)


# Evaluation

Running the cell below lets you check how many points your solution would receive on the validation data. Before submitting, make sure the whole notebook runs from start to finish without errors and without requiring user intervention after selecting "Run All".


In [ ]:
######################### DO NOT CHANGE THIS CELL ##########################

if not FINAL_EVALUATION_MODE:
    embedder = Embedder()

    with torch.no_grad():
        results = search_topk_texts(embedder, corpus, queries, top_k=10)

    # Compute nDCG
    ndcg = evaluate_retrieval_ndcg(matching_texts, results)

    # Compute final score from nDCG
    points = compute_score(ndcg)

    print(f"\nNumber of queries: {len(queries)}")
    print(f"Number of texts: {len(corpus)}")
    print(f"nDCG: {ndcg:.3f}")
    print(f"Point score: {points}")


During grading, the model will be saved as `your_model.pkl` and evaluated on the test set.


In [8]:
######################### DO NOT CHANGE THIS CELL ##########################

if FINAL_EVALUATION_MODE:
    import cloudpickle

    OUTPUT_PATH = "file_output"
    FUNCTION_FILENAME = "your_model.pkl"
    FUNCTION_OUTPUT_PATH = os.path.join(OUTPUT_PATH, FUNCTION_FILENAME)

    if not os.path.exists(OUTPUT_PATH):
        os.makedirs(OUTPUT_PATH)

    with open(FUNCTION_OUTPUT_PATH, "wb") as f:
        cloudpickle.dump(Embedder, f)
